In [1]:
%matplotlib qt
import numpy as np
import matplotlib.pyplot as plt
from astropy.io import fits
from datetime import datetime
import glob
from reprojection import *
from show import *

In [4]:
files = np.array(sorted(glob.glob('/home/ulyanov/data/solo/phi/2026/blos_/*')))
#files = np.array(sorted(glob.glob('C:/Users/ikiru/data/4science/blos/*')))
#files = np.array(sorted(glob.glob('/home/ulyanov/data/sdo/hmi/2315/*')))
dates = np.array([datetime.fromisoformat(file.split('/')[-1].split('_')[-3]) for file in files])

#t = np.where(np.all([dates > datetime(2026,7,22),
#                     dates < datetime(2026,8,30)], axis=0))[0] # CR2314

#t = np.where(np.all([dates > datetime(2026,8,30),
#                     dates < datetime(2026,9,29)], axis=0))[0] # CR2315


t = np.where(np.all([dates > datetime(2026,7,22),
                     dates < datetime(2026,9,29)], axis=0))[0] # CR2314 + CR2315

files = files[t]
files[0], files[-1], len(files)

(np.str_('/home/ulyanov/data/solo/phi/2026/blos_/phi-fdt-blos_20260722T030009_V202610210936C_0647220501.fits'),
 np.str_('/home/ulyanov/data/solo/phi/2026/blos_/phi-fdt-blos_20260928T213003_V202610202220C_0649280507.fits'),
 215)

In [ ]:
data, view = make_map(files[:], correct_dr=False, kind='synoptic', minmu=0.1, wsid=13, sine=False, full_output=True)

In [ ]:
fig, ax = show_data(data[0], view, cmap='seismic', vmin=-20, vmax=20)

In [ ]:
plt.figure(figsize=(10,8))
plt.plot(np.nanstd(data[0], axis=1))
plt.plot(np.nanpercentile(data[0], 70, axis=1) - np.nanpercentile(data[0], 30, axis=1))

In [5]:
Q = []
wsid_range = np.arange(9, 16, 0.1)
for wsid in wsid_range:
    print(wsid)

    data, view = make_map(files[:], correct_dr=False, kind='synoptic', minmu=0.1, wsid=wsid, sine=False, full_output=True)
    #Q += [np.nanstd(data[0] / np.sqrt(data[1]), axis=1)]
    #Q += [np.nanpercentile(data[0], 70, axis=1) - np.nanpercentile(data[0], 30, axis=1)]
    Q += [data]

Q = np.array(Q)
np.savez('Q.npz', Q=Q)

9.0


/home/ulyanov/PycharmProjects/reprojection/src/reprojection.py:98: RuntimeWarning: invalid value encountered in divide
  w = np.expand_dims(n / N, (-2, -1))
/home/ulyanov/PycharmProjects/reprojection/src/reprojection.py:135: RuntimeWarning: invalid value encountered in divide
  S = (FF - GF[:, :, -1:, :] ** 2 - Q ** 2 * (LL - GL[:, :, -1:, :] ** 2)) / LL / np.expand_dims(N - 1, (-1, -2))


9.1
9.2
9.299999999999999
9.399999999999999
9.499999999999998
9.599999999999998
9.699999999999998
9.799999999999997
9.899999999999997
9.999999999999996
10.099999999999996
10.199999999999996
10.299999999999995
10.399999999999995
10.499999999999995
10.599999999999994
10.699999999999994
10.799999999999994
10.899999999999993
10.999999999999993
11.099999999999993
11.199999999999992
11.299999999999992
11.399999999999991
11.499999999999991
11.59999999999999
11.69999999999999
11.79999999999999
11.89999999999999
11.99999999999999
12.099999999999989
12.199999999999989
12.299999999999988
12.399999999999988
12.499999999999988
12.599999999999987
12.699999999999987
12.799999999999986
12.899999999999986
12.999999999999986
13.099999999999985
13.199999999999985
13.299999999999985
13.399999999999984
13.499999999999984
13.599999999999984
13.699999999999983
13.799999999999983
13.899999999999983
13.999999999999982
14.099999999999982
14.199999999999982
14.299999999999981
14.39999999999998
14.49999999999998


In [9]:
nx = Q.shape[-2]
lat = (np.arange(nx) / (nx - 1) * 2 - 1) * 90
#np.savez('diff_rot.npz', data=Q, wsid=wsid_range, lat=lat)

In [29]:
from utils import rebin

s = np.load('diff_rot.npz')
Q = s['data']
wsid_range = s['wsid']
lat = s['lat']

Q = rebin(Q, 4, axis=2)
lat = lat[::4] + 1.5


In [30]:
Q.shape

(70, 3, 180, 1800)

In [31]:
A, B, C = 14.712, -2.396, -1.787

#q = np.nanpercentile(Q[:,0], 95, axis=-1) - np.nanpercentile(Q[:,0], 5, axis=-1)

#w = Q[:,-1] #** 2
##m = np.nanmean(Q[:,0] * w, axis=-1) / np.nanmean(w, axis=-1)
#s = np.nanmean(Q[:,0] ** 2 * w, axis=-1) / np.nanmean(w, axis=-1)
#q = s - m ** 2
q = np.nanstd(Q[:,0] / np.sqrt(Q[:,1]), axis=-1)
q = np.nanargmax(q, axis=0)

y = np.sin(lat * np.pi / 180)

plt.figure(figsize=(10,8))
plt.plot(lat, wsid_range[q])
plt.plot(lat, A + B * y ** 2 + C * y ** 4)
plt.tight_layout()

In [40]:
plt.figure(figsize=(10,8))
plt.imshow(np.abs(Q[0,0]) / np.sqrt(Q[0,1]), origin='lower', aspect='auto', cmap='seismic', interpolation='none', vmin=0, vmax=5)
plt.tight_layout()

In [39]:
plt.figure(figsize=(10,8))
plt.imshow(Q[0,0], origin='lower', aspect='auto', cmap='seismic', interpolation='none', vmin=-25, vmax=25)
plt.tight_layout()